In [0]:
import json
import sys
from databricks.sdk import WorkspaceClient

sys.path.append('..')
from resources.brick_setup_functions import AgentBricksManager

# Load configuration (includes sa_name)
exec(open('../config.py').read())

w = WorkspaceClient()
brick_manager = AgentBricksManager(w)

# Step 1: Look up the Supervisor Agent tile_id by name
print(f"🔍 Looking up Supervisor Agent '{sa_name}'...")
resp = w.api_client.do(
    'GET',
    f'/api/2.0/tiles?filter=name_contains%3D{sa_name}%26%26tile_type%3DMAS'
)
tiles = [t for t in resp.get('tiles', []) if t.get('name') == sa_name]
assert tiles, f"Supervisor Agent '{sa_name}' not found. Please create it first."
mas_tile_id = tiles[0]['tile_id']
print(f"   Found tile_id: {mas_tile_id}")

# Step 2: Get current MAS configuration
mas_data = brick_manager.mas_get(mas_tile_id)
mas_info = mas_data['multi_agent_supervisor']
current_agents = mas_info.get('agents', [])
print(f"   Current agents ({len(current_agents)}):")
for i, a in enumerate(current_agents):
    print(f"     {i+1}. {a['name']} ({a['agent_type']})")

# Step 3: Check if 'you-mcp' is already added
already_exists = any(
    a.get('external_mcp_server', {}).get('connection_name') == 'you-mcp'
    for a in current_agents
)

if already_exists:
    print("\n✅ 'you-mcp' MCP server is already added to the supervisor.")
else:
    # Step 4: Add the new MCP server agent
    mcp_agent = {
        "name": "You_Web_Search",
        "description": "Web search tool powered by You.com MCP server. Use for searching the internet for real-time information, news, and general knowledge queries.",
        "agent_type": "external-mcp-server",
        "external_mcp_server": {"connection_name": "you-mcp"}
    }

    updated_agents = current_agents + [mcp_agent]

    print(f"\n🤖 Adding 'you-mcp' MCP server as '{mcp_agent['name']}'...")
    result = brick_manager.mas_update(
        tile_id=mas_tile_id,
        name=sa_name,
        agents=updated_agents
    )
    print(f"✅ Successfully added 'you-mcp' to {sa_name}!")

    # Step 5: Verify
    updated_mas = brick_manager.mas_get(mas_tile_id)
    updated_agents_list = updated_mas['multi_agent_supervisor'].get('agents', [])
    print(f"\n📋 Updated agents ({len(updated_agents_list)}):")
    for i, a in enumerate(updated_agents_list):
        print(f"   {i+1}. {a['name']} ({a['agent_type']})")
        if 'external_mcp_server' in a:
            print(f"      Connection: {a['external_mcp_server'].get('connection_name')}")
        elif 'serving_endpoint' in a:
            print(f"      Endpoint: {a['serving_endpoint']['name']}")
        elif 'genie_space' in a:
            print(f"      Genie ID: {a['genie_space']['id']}")
        elif 'unity_catalog_function' in a:
            uc = a['unity_catalog_function']['uc_path']
            print(f"      Function: {uc['catalog']}.{uc['schema']}.{uc['name']}")